# 05 - Evaluation: Robustness & Kualitas Model LDA## TujuanMenguji seberapa konsisten model LDA terhadap variasi acak (multi-seed),membandingkan K, dan mengukur kualitas serta diversitas topik.## Asal (Input)`model/lda_model.gensim` + `data/intermediate/` (korpus 285 dokumen)## Keluaran (Output)`experiments/` - hasil kestabilan multi-seed, sweep K, metrics## Lima Uji yang Dilakukan- **A.** Stabilitas multi-seed (K=8, 10 seed) - C_V & U_Mass- **B.** Sweep K (4-12, 3 seed per K)- **C.** Perplexity train vs hold-out- **D.** Kualitas & diversitas topik- **E.** Distribusi theta & proporsi per tahun

In [ ]:
import sys, io as _iofrom pathlib import Path_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()sys.path.insert(0, str(_ROOT))try:    sys.stdout.reconfigure(encoding='utf-8', errors='replace')except Exception:    passimport pandas as pd, numpy as npfrom src import config as CC.ensure_dirs()print(C.describe())

## Cara Menjalankan Evaluasi KomprehensifUji lengkap (multi-seed 10 seed, sweep K 4-12, perplexity) dijalankan lewatmodul oriented-script:```bashpython -m src.evaluation_lda```Modul tersebut butuh sekitar 10 menit. Notebook ini menampilkan ringkasancepat metrik utama yang dapat dihitung langsung dari model tersimpan.

In [ ]:
# Ringkasan cepat tanpa menunggu 10 seed penuhimport ast, numpy as npfrom gensim import corporafrom gensim.models import LdaModel, CoherenceModelprep = pd.read_csv(C.PREPROCESSED_CSV)texts = prep['tokens'].apply(ast.literal_eval).tolist()d = corpora.Dictionary.load(str(C.DICTIONARY_PATH))m = LdaModel.load(str(C.LDA_MODEL))corpus = [d.doc2bow(t) for t in texts]# Topik Diversityt10 = [w for t in range(m.num_topics) for w,_ in m.show_topic(t, topn=10)]t15 = [w for t in range(m.num_topics) for w,_ in m.show_topic(t, topn=15)]print(f'Topic Diversity Top-10 : {len(set(t10))/len(t10):.4f}')print(f'Topic Diversity Top-15 : {len(set(t15))/len(t15):.4f}')# Theta distributiontheta = np.zeros((len(corpus), m.num_topics))for i, c in enumerate(corpus):    for tid, p in m.get_document_topics(c, minimum_probability=0.0):        theta[i, tid] = pmx = theta.max(axis=1)print(f'Dokumen theta < 0.40   : {np.mean(mx<0.40)*100:.2f}%')print(f'Dokumen theta < 0.50   : {np.mean(mx<0.50)*100:.2f}%')

## Langkah 2 - Perplexity Train vs Hold-out

In [ ]:
from sklearn.model_selection import train_test_splitfrom gensim.models import LdaModel as LMtr_texts, te_texts, _, _ = train_test_split(    texts, texts, test_size=C.TRAIN_TEST_SPLIT, random_state=C.RANDOM_SEED)tr_corpus = [d.doc2bow(t) for t in tr_texts]te_corpus = [d.doc2bow(t) for t in te_texts]m_eval = LM(corpus=tr_corpus, id2word=d, num_topics=C.NUM_TOPICS,            alpha=C.LDA_ALPHA, eta=C.LDA_ETA,            passes=C.LDA_PASSES, iterations=C.LDA_ITERATIONS,            random_state=C.RANDOM_SEED)print(f'Train log perplexity   : {m_eval.log_perplexity(tr_corpus):.4f}')print(f'Hold-out log perplexity: {m_eval.log_perplexity(te_corpus):.4f}')

## Langkah 3 - Elbow Curve Sweep K

In [ ]:
# Jalankan sweep penuh via modul (lebih lambat, ~10 menit):#   python -m src.evaluation_lda## Contoh visualisasi elbow dari data yang sudah ada:print('Jalankan: python -m src.evaluation_lda')print('Hasil disimpan di experiments/k_sweep.csv')

## Ringkasan Fase 05 - Evaluation| Metrik | Nilai ||:---|---:|| Topic Diversity (Top-10) | 0,7125 || Topic Diversity (Top-15) | 0,6917 || Dokumen ambigu (theta < 0.50) | 13,33% |Diversitas topik yang relatif tinggi menunjukkan kedelapan topik memangberbeda satu sama lain, bukan sekadar pembagian arsip kata yang sama.

## Peta Artefak| Artefak | Dibuat oleh | Consumed by ||:---|:---|:---|| `experiments/*` | Notebook 05 / src.evaluation_lda | Lampiran Bab IV |